# VietMedBridge — 01d: Thử lại có robots guard và pacing

Chạy trong **Colab CPU mới** sau 01c. Notebook này đọc experiment 01c đã
khóa bằng SHA-256, chỉ thử lại 11 URL chưa tạo article candidate. Mỗi
request, redirect và retry đều kiểm robots và áp dụng Crawl-delay hoặc
Request-rate. Không bật browser, không vượt Disallow/robots lỗi, không
tự nhập kết quả vào Stage A. Mỗi ID có marker trên Drive để resume.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "chọn phiên bản runtime có Python trong khoảng này."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
# Chạy lại cell này sau MỖI runtime mới/restart: package chỉ được cài trong phiên Colab.
drive.mount("/content/drive")

# Giữ cùng DATA_ROOT trong cả bốn notebook.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
EXPECTED_PIPELINE_API = 2
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "guarded_recovery_code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
if lock and lock.get("pipeline_api") != EXPECTED_PIPELINE_API and not CODE_REVISION:
    raise RuntimeError(
        "DATA_ROOT đang dùng code cũ. Để nâng lên data-v2, đặt CODE_REVISION='main' "
        "ở cell này rồi chạy lại; dùng tên crawl/build run mới. Raw cũ vẫn được giữ."
    )
reference = CODE_REVISION or lock.get("git_commit") or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if "vietmedbridge" in sys.modules and globals().get("_VMB_IMPORTED_COMMIT") != CODE_COMMIT:
    raise RuntimeError("Mã nguồn đổi trong runtime đã import package. Restart session rồi chạy lại notebook.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    cwd=CHECKOUT,
    check=True,
)
PACKAGE_SOURCE = CHECKOUT / "src"
if not (PACKAGE_SOURCE / "vietmedbridge" / "__init__.py").is_file():
    raise RuntimeError(
        f"Repo checkout thiếu package source: {PACKAGE_SOURCE}; commit={CODE_COMMIT}. "
        "Kiểm tra lại git fetch/checkout ở output của cell Bootstrap."
    )
sys.path.insert(0, str(PACKAGE_SOURCE))
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json, runtime_versions
from vietmedbridge import PIPELINE_API_VERSION
if PIPELINE_API_VERSION != EXPECTED_PIPELINE_API:
    raise RuntimeError("Notebook và package API không khớp; chọn commit data-v2 và restart session.")
_VMB_IMPORTED_COMMIT = CODE_COMMIT
if not lock or CODE_REVISION:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT, "pipeline_api": PIPELINE_API_VERSION})
PIPELINE_CONFIG = json.loads((CHECKOUT / "configs/data_pipeline.json").read_text())
atomic_json(DATA_ROOT / "runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
import vietmedbridge
print("Package source:", Path(vietmedbridge.__file__).resolve())
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 1. Kiểm input 01c và lập danh sách 11 URL


In [ ]:
import pandas as pd
from vietmedbridge.artifacts import (
    atomic_json, code_fingerprint, digest_json, read_json, sha256_file, utc_now,
)

SOURCE_KEY = "6a5793cdc03be0cb"
EXPECTED_ATTEMPTS_SHA256 = "09f3edd03c459b8791bb819805656e01626a837eec898c2a1d46130af5de5244"
EXPECTED_ROBOTS_SHA256 = "637738e9641c7b8cfa668b9f30ea547e9de664fe8a9204458e6836bd023b3a27"
REPORT_DIR = DATA_ROOT / "reports/crawl_recovery/stage-a-v2"
SOURCE_DIR = REPORT_DIR / "experiments" / SOURCE_KEY
SOURCE = read_json(SOURCE_DIR / "summary.json")
SOURCE_EXPERIMENT = read_json(SOURCE_DIR / "experiment.json")
TRIAGE = read_json(REPORT_DIR / "triage.json")
if SOURCE_EXPERIMENT["source_run_signature"] != TRIAGE["run_signature"]:
    raise ValueError("01c và 01b không cùng Stage A run.")
if SOURCE["attempts_sha256"] != EXPECTED_ATTEMPTS_SHA256 or SOURCE["robots_sha256"] != EXPECTED_ROBOTS_SHA256:
    raise ValueError("01c summary khác experiment đã báo; xác minh key và hash trước khi chạy.")
ATTEMPTS_PATH = DATA_ROOT / SOURCE["attempts_csv"]
ROBOTS_PATH = DATA_ROOT / SOURCE["robots_csv"]
if sha256_file(ATTEMPTS_PATH) != EXPECTED_ATTEMPTS_SHA256 or sha256_file(ROBOTS_PATH) != EXPECTED_ROBOTS_SHA256:
    raise ValueError("CSV 01c đã đổi hoặc tải thiếu.")
ATTEMPTS = pd.read_csv(ATTEMPTS_PATH, dtype={"doc_id": "int64", "url": "string"})
ROBOTS = pd.read_csv(ROBOTS_PATH, dtype={"doc_id": "int64", "url": "string"})
if len(ROBOTS) != SOURCE["input_unresolved"] or ROBOTS["doc_id"].duplicated().any():
    raise ValueError("robots.csv không đủ 83 official IDs hoặc có ID trùng.")
if len(ATTEMPTS) != SOURCE["attempts"] or ATTEMPTS[["doc_id", "method"]].duplicated().any():
    raise ValueError("attempts.csv không đủ 15 thử nghiệm hoặc có ID/method trùng.")
ATTEMPTS["article_candidate"] = ATTEMPTS["article_candidate"].astype(str).str.lower().eq("true")
ROBOTS["robots_allowed"] = ROBOTS["robots_allowed"].astype(str).str.lower().eq("true")
official = ROBOTS.set_index("doc_id")
for row in ATTEMPTS.itertuples(index=False):
    if row.doc_id not in official.index or row.url != official.loc[row.doc_id, "url"]:
        raise ValueError(f"ID/URL trong attempts không khớp robots: {row.doc_id}")
    if not bool(official.loc[row.doc_id, "robots_allowed"]):
        raise ValueError(f"01c đã thử URL không được phép: {row.doc_id}")
if int(ATTEMPTS["article_candidate"].sum()) != SOURCE["article_candidates_needing_human_review"]:
    raise ValueError("Số article candidate khác summary.json.")
REMAINING = ATTEMPTS[~ATTEMPTS["article_candidate"]].copy().sort_values("doc_id")
print("01c attempts:", len(ATTEMPTS), "| cần thử lại:", len(REMAINING))
display(REMAINING.reindex(columns=[
    "doc_id", "url", "original_error", "outcome", "http_status", "error_type",
]).fillna(""))
print("4 article candidates từ 01c vẫn cần human review; notebook này không fetch lại chúng.")


## 2. Thử Scrapling HTTP với robots mới, pacing và redirect guard


In [ ]:
import importlib.metadata
import subprocess
import time
from urllib.parse import urlsplit
from vietmedbridge.artifacts import publish_file
from vietmedbridge.crawl import CrawlConfig, Fetcher
from vietmedbridge.recovery import RecoveryConfig, guarded_http_fetch
from vietmedbridge.text import EXPECTED_PARSE_ERRORS, extract_source
from vietmedbridge.quality import document_quality

if importlib.metadata.packages_distributions().get("scrapling") is None or (
    importlib.metadata.version("scrapling") != "0.4.15"
):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "scrapling[fetchers]==0.4.15"], check=True)
from scrapling.fetchers import Fetcher as ScraplingFetcher

FOLLOWUP_KEY = digest_json({
    "source_key": SOURCE_KEY, "attempts_sha256": EXPECTED_ATTEMPTS_SHA256,
    "code_sha256": code_fingerprint(), "policy": "guarded-http-v1",
    "scrapling_version": "0.4.15",
})[:16]
FOLLOWUP_DIR = REPORT_DIR / "experiments" / FOLLOWUP_KEY
MARKERS_DIR = FOLLOWUP_DIR / "markers"
ASSETS_DIR = FOLLOWUP_DIR / "assets"
MARKERS_DIR.mkdir(parents=True, exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = FOLLOWUP_DIR / "experiment.json"
manifest = {"source_key": SOURCE_KEY, "source_attempts_sha256": EXPECTED_ATTEMPTS_SHA256,
            "code_commit": CODE_COMMIT, "code_sha256": code_fingerprint(),
            "policy": "guarded-http-v1", "input_ids": len(REMAINING)}
if manifest_path.exists() and read_json(manifest_path) != manifest:
    raise ValueError("Follow-up checkpoint khác cấu hình hiện tại.")
if not manifest_path.exists():
    atomic_json(manifest_path, manifest)

USER_AGENT = "VietMedBridge/0.2 (+https://github.com/Platypus27-coder/VietMedBridge)"
guard = Fetcher(CrawlConfig(concurrency=1, per_host_delay=3, attempts=1))
settings = RecoveryConfig(attempts=2, max_redirects=5, max_body_bytes=16 * 1024 * 1024)
held_hosts = set()
try:
    for row in REMAINING.itertuples(index=False):
        marker = MARKERS_DIR / f"{int(row.doc_id)}.json"
        if marker.exists():
            saved = read_json(marker)
            if saved["url"] != row.url or saved["source_attempt_outcome"] != row.outcome:
                raise ValueError(f"Marker khác URL/outcome gốc: {row.doc_id}")
            if saved.get("outcome") in ("rate_limited_retry_later", "server_retry_later", "host_rate_limit_hold"):
                held_hosts.add(urlsplit(saved.get("final_url", row.url)).hostname)
            continue
        started = time.monotonic()
        record = {"doc_id": int(row.doc_id), "url": row.url,
                  "source_attempt_outcome": row.outcome,
                  "method": "guarded_scrapling_http", "started_at": utc_now(),
                  "article_candidate": False}
        if urlsplit(row.url).hostname in held_hosts:
            record["outcome"] = "host_rate_limit_hold"
        else:
            result, body = await guarded_http_fetch(
                row.url, guard, ScraplingFetcher.get,
                user_agent=USER_AGENT, config=settings, held_hosts=held_hosts,
            )
            record.update(result)
            if body is not None:
                local_asset = WORK_DIR / f"guarded-recovery-{int(row.doc_id)}.bin"
                local_asset.write_bytes(body)
                target = ASSETS_DIR / local_asset.name
                try:
                    record["body_sha256"] = publish_file(local_asset, target)
                finally:
                    local_asset.unlink(missing_ok=True)
                record["asset_path"] = str(target.relative_to(DATA_ROOT))
                try:
                    extracted = extract_source(body, record["content_type"])
                    quality = document_quality(
                        extracted["source_text"], title=extracted["title"],
                        raw_bytes=len(body), content_type=record["content_type"],
                    )
                    record.update(
                        source_text_sha256=extracted["source_text_sha256"],
                        source_chars=len(extracted["source_text"]),
                        source_preview=extracted["source_text"][:500],
                        quality_tier=quality["quality_tier"],
                        quality_flags=quality["quality_flags"],
                        article_candidate=len(extracted["source_text"]) >= 80,
                    )
                    record["outcome"] = ("article_candidate" if record["article_candidate"]
                                         else "short_text_review")
                except EXPECTED_PARSE_ERRORS as exc:
                    record.update(outcome="extract_rejected", extract_error=str(exc)[:300])
        record.update(elapsed_ms=round((time.monotonic() - started) * 1000),
                      finished_at=utc_now())
        atomic_json(marker, record)
        print(row.doc_id, record["outcome"])
finally:
    await guard.client.aclose()
print("Follow-up experiment:", FOLLOWUP_DIR)


## 3. Xuất kết quả; không nhập vào Stage A


In [ ]:
records = [read_json(MARKERS_DIR / f"{int(i)}.json") for i in REMAINING["doc_id"]]
result_path = FOLLOWUP_DIR / "attempts.csv"
pd.DataFrame(records).to_csv(result_path, index=False, encoding="utf-8-sig")
summary = {"source_experiment": SOURCE_KEY, "input_non_candidates": len(REMAINING),
           "outcomes": pd.Series([r["outcome"] for r in records]).value_counts().to_dict(),
           "new_article_candidates_needing_human_review": sum(bool(r.get("article_candidate")) for r in records),
           "attempts_csv": str(result_path.relative_to(DATA_ROOT)),
           "attempts_sha256": sha256_file(result_path), "created_at": utc_now()}
atomic_json(FOLLOWUP_DIR / "summary.json", summary)
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(pd.DataFrame(records).reindex(columns=[
    "doc_id", "source_attempt_outcome", "outcome", "http_status",
    "error_type", "article_candidate", "elapsed_ms",
]).fillna(""))
print("Gửi attempts.csv và summary.json của experiment này để audit nội dung.")
